# cAPTure: development OOF operational evaluation

This notebook compares XGB-P, both context ablations, and full XGB-P+T under the same predeclared false-alert budgets. It reads only checksum-verified development OOF artifacts. A separate score threshold is selected for each model and budget from development OOF scores; held-out author-train and final-test scenario contents are not read.


## 1. Prepare the Colab environment


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"],
                                 cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/requirements-capture-xgb.txt",
                  "code/python/utils/capture_xgb_p_t.py",
                  "code/python/utils/capture_oof_operational.py",
                  "configs/capture_experiment_v1.yaml"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


## 2. Bind the completed model runs

Enter the completed XGB-P+T and ablation run IDs. Set `EVALUATION_RUN_ID` only when resuming an existing evaluation. The evaluator validates fold assignments, artifact checksums, packet counts, context provenance, and the frozen manifest policy before reporting metrics.


In [ ]:
from utils.capture_oof_operational import (
    run_operational_oof_evaluation, validate_operational_run,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
BASELINE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p"
PRIMARY_RUN_ID = None  # Set to the completed XGB-P+T run ID.
ABLATION_RUN_ID = None  # Set to the completed ablation run ID.
EVALUATION_RUN_ID = None  # Set only when resuming an existing evaluation.
if PRIMARY_RUN_ID is None or ABLATION_RUN_ID is None:
    raise ValueError("Set PRIMARY_RUN_ID and ABLATION_RUN_ID before evaluating OOF runs.")
if EVALUATION_RUN_ID is None:
    EVALUATION_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_operational_oof"
ABLATION_DIR = DRIVE_ROOT / "xgb_p_t_ablation_runs" / ABLATION_RUN_ID
RUN_DIRS = {
    "xgb_p": BASELINE_RUN_DIR,
    "current_window": ABLATION_DIR / "current_window",
    "history": ABLATION_DIR / "history",
    "full": DRIVE_ROOT / "xgb_p_t_runs" / PRIMARY_RUN_ID,
}
OUTPUT_DIR = DRIVE_ROOT / "operational_oof_runs" / EVALUATION_RUN_ID
print("Input runs:", RUN_DIRS)
print("Evaluation output:", OUTPUT_DIR)


## 3. Compute or verify the immutable development report

The primary budget is one false-alert window per hour. Sensitivity budgets are one per 12 hours and one per five minutes. A window alerts when its maximum packet score is at least the model threshold. Only windows without malicious packets contribute false alerts and benign exposure; empty wall-clock windows contribute exposure but cannot alert. The threshold is the most permissive one whose worst fold mean scenario rate meets the budget. One attack-step iteration is detected only when one of its malicious packets crosses the threshold, at that packet window's close.


In [ ]:
if OUTPUT_DIR.exists():
    report = validate_operational_run(OUTPUT_DIR, MANIFEST_PATH, RUN_DIRS)
else:
    report = run_operational_oof_evaluation(
        manifest_path=MANIFEST_PATH, run_dirs=RUN_DIRS,
        output_dir=OUTPUT_DIR, batch_size=250_000)
print("Evaluation run ID:", EVALUATION_RUN_ID)
print("Budgets per hour:", report["budgets_per_hour"])


## 4. Review thresholds and primary-budget metrics

Compare sequence detection rate and latency alongside false alerts. The detected-only latency is diagnostic; the main latency summary retains missed iterations using their attack-step duration. Review individual scenarios before interpreting the hierarchical macro result.


In [ ]:
budget_name = "one_per_hour"
threshold_rows = []
macro_rows = []
scenario_rows = []
for model_name, model_report in report["models"].items():
    selected = model_report["thresholds"][budget_name]
    threshold_rows.append({
        "model": model_name,
        "score_threshold": selected["threshold"],
        "worst_fold_false_alert_windows_per_hour": selected[
            "worst_fold_false_alert_windows_per_hour"],
    })
    result = model_report["budgets"][budget_name]
    macro_rows.append({"model": model_name, **result["hierarchical_macro"]})
    for scenario, metrics in result["scenario_metrics"].items():
        scenario_rows.append({"model": model_name, "scenario": scenario, **metrics})
display(pd.DataFrame(threshold_rows).set_index("model"))
display(pd.DataFrame(macro_rows).set_index("model"))
display(pd.DataFrame(scenario_rows).set_index(["model", "scenario"])[[
    "fold", "false_alert_windows_per_hour", "packet_recall",
    "sequence_detection_rate", "mean_miss_capped_latency_seconds",
    "attack_step_iterations", "detected_iterations"]])


## 5. Review budget sensitivity and weak attack steps

The two sensitivity budgets use thresholds selected by the same rule. Missed iterations remain in the saved report for independent review.


In [ ]:
sensitivity_rows = []
for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        selected = model_report["thresholds"][budget_name]
        summary = model_report["budgets"][budget_name]["hierarchical_macro"]
        sensitivity_rows.append({"model": model_name, "budget": budget_name,
                                 "threshold": selected["threshold"], **summary})
display(pd.DataFrame(sensitivity_rows).set_index(["model", "budget"]))
step_rows = []
for model_name, model_report in report["models"].items():
    for step in model_report["budgets"]["one_per_hour"]["step_metrics"].values():
        step_rows.append({"model": model_name, **step})
display(pd.DataFrame(step_rows).sort_values(
    ["sequence_detection_rate", "mean_miss_capped_latency_seconds"],
    ascending=[True, False]).head(40))
